# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math, statistics
from pathlib import Path
import numpy as np, torch

# Hỗ trợ chạy từ thư mục repo hoặc từ submission_<MSSV>/code/.
_repo_candidate = Path("../..").resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    _repo_candidate = Path.cwd().resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    raise FileNotFoundError("Không tìm thấy repo root có scripts/split_data.py; hãy đặt REPO_ROOT đúng.")
REPO_ROOT = str(_repo_candidate)
OUT_DIR = str(Path(REPO_ROOT) / "submission_2A202602793")
CODE_DIR = str(Path(OUT_DIR) / "code")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# Colab: clone repo rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-NeuralNetwork".
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-NeuralNetwork.git

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Thiết bị: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, load_split, make_val_split, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


Phiên bản PyTorch: 2.6.0
Thiết bị: cpu


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
split_process = subprocess.run(
    [sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True,
    text=True, capture_output=True
)
print(split_process.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

X_train_full, y_train_full, X_eval_raw, y_eval_raw, row_ids_raw = load_split(f"{REPO_ROOT}/data/processed")
X_tr_raw, y_tr_raw, X_val_raw, y_val_raw = make_val_split(X_train_full, y_train_full, val_fraction=0.2, seed=42)
for split_name, features, labels in (
    ("train", data["X_tr"], data["y_tr"]),
    ("val", data["X_val"], data["y_val"]),
    ("eval", data["X_eval"], data["y_eval"]),
):
    print(f"{split_name}: X={tuple(features.shape)} {features.dtype}, y={tuple(labels.shape)} {labels.dtype}, nhãn={int(labels.min())}..{int(labels.max())}")
print(f"eval_row_id: {data['eval_row_id'].shape}, dtype={data['eval_row_id'].dtype}")

train_numeric = data["X_tr"][:, :10].detach().cpu().numpy().astype(np.float64)
numeric_mean = train_numeric.mean(axis=0)
numeric_std = train_numeric.std(axis=0)
print("Trung bình 10 cột số trên X_tr:", np.round(numeric_mean, 7))
print("Độ lệch chuẩn 10 cột số trên X_tr:", np.round(numeric_std, 7))
assert np.allclose(numeric_mean, 0.0, atol=1e-5)
assert np.allclose(numeric_std, 1.0, atol=1e-4)
assert np.array_equal(data["X_tr"][:, 10:].detach().cpu().numpy(), X_tr_raw[:, 10:])
assert tuple(data["X_tr"].shape) == (371847, 54)
assert tuple(data["X_val"].shape) == (92962, 54)
assert tuple(data["X_eval"].shape) == (116203, 54)
assert len(data["eval_row_id"]) == 116203 and len(np.unique(data["eval_row_id"])) == 116203

majority_class = int(np.bincount(y_tr_raw, minlength=7).argmax())
majority_val_acc = float(np.mean(y_val_raw == majority_class))

_small_X = torch.zeros((1000, 54), dtype=torch.float32, device=device)
_small_y = torch.zeros(1000, dtype=torch.int64, device=device)
_batch_sizes = [len(yb) for _, yb in iterate_batches(_small_X, _small_y, 512, shuffle=False)]
print(f"Kích thước batch minh hoạ (N=1000, batch=512): {_batch_sizes}")
assert _batch_sizes == [512, 488]

train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data/processed/train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data/processed/eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876



X_tr=(371847, 54), X_val=(92962, 54), X_eval=(116203, 54)
val accuracy (luôn đoán lớp đa số train 1) = 0.4876


train: X=(371847, 54) torch.float32, y=(371847,) torch.int64, nhãn=0..6
val: X=(92962, 54) torch.float32, y=(92962,) torch.int64, nhãn=0..6
eval: X=(116203, 54) torch.float32, y=(116203,) torch.int64, nhãn=0..6
eval_row_id: (116203,), dtype=int64
Trung bình 10 cột số trên X_tr: [-2.e-07 -0.e+00  1.e-07  0.e+00  0.e+00  0.e+00 -2.e-07 -4.e-07 -1.e-07
  0.e+00]
Độ lệch chuẩn 10 cột số trên X_tr: [1.0000001 1.        1.        1.        1.        1.        1.
 1.        1.        1.       ]
Kích thước batch minh hoạ (N=1000, batch=512): [512, 488]


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
# TODO 1: model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device); assert count_params(model) == EXPECTED_PARAMS[(256,128)]
# TODO 2: cho một lô ngẫu nhiên (8, 54) chạy qua; in shape; assert logits.shape == (8, 7)
# TODO 3: loss bước 0 trên val (eval mode) ≈ ln 7 ≈ 1.946  -> in giá trị đo và nhận xét
# TODO 4: quá khớp 20 mẫu (tắt dropout, vài trăm bước) -> loss về gần 0; vẽ đường cong loss
# TODO 5: sau một lần backward, in grad norm của từng tham số; assert tất cả khác None và khác 0

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [5]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [6]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [7]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [8]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [9]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary